# Part 3: Single-View Geometry

## Usage
This code snippet provides an overall code structure and some interactive plot interfaces for the *Single-View Geometry* section of Assignment 3. In [main function](#Main-function), we outline the required functionalities step by step. Some of the functions which involves interactive plots are already provided, but [the rest](#Your-implementation) are left for you to implement.

## Package installation
- In this code, we use `tkinter` package. Installation instruction can be found [here](https://anaconda.org/anaconda/tk).

# Common imports

In [ ]:
%matplotlib tk
import matplotlib.pyplot as plt
import numpy as np
import math
from PIL import Image

# Provided functions

In [ ]:
def get_input_lines(im, min_lines=3):
    """
    Allows user to input line segments; computes centers and directions.
    Inputs:
        im: np.ndarray of shape (height, width, 3)
        min_lines: minimum number of lines required
    Returns:
        n: number of lines from input
        lines: np.ndarray of shape (3, n)
            where each column denotes the parameters of the line equation
        centers: np.ndarray of shape (3, n)
            where each column denotes the homogeneous coordinates of the centers
    """
    n = 0
    lines = np.zeros((3, 0))
    centers = np.zeros((3, 0))

    plt.figure()
    plt.imshow(im)
    plt.show()
    print('Set at least %d lines to compute vanishing point' % min_lines)
    while True:
        print('Click the two endpoints, use the right key to undo, and use the middle key to stop input')
        clicked = plt.ginput(2, timeout=0, show_clicks=True)
        if not clicked or len(clicked) < 2:
            if n < min_lines:
                print('Need at least %d lines, you have %d now' % (min_lines, n))
                continue
            else:
                # Stop getting lines if number of lines is enough
                break

        # Unpack user inputs and save as homogeneous coordinates
        pt1 = np.array([clicked[0][0], clicked[0][1], 1])
        pt2 = np.array([clicked[1][0], clicked[1][1], 1])
        # Get line equation using cross product
        # Line equation: line[0] * x + line[1] * y + line[2] = 0
        line = np.cross(pt1, pt2)
        lines = np.append(lines, line.reshape((3, 1)), axis=1)
        # Get center coordinate of the line segment
        center = (pt1 + pt2) / 2
        centers = np.append(centers, center.reshape((3, 1)), axis=1)

        # Plot line segment
        plt.plot([pt1[0], pt2[0]], [pt1[1], pt2[1]], color='b')

        n += 1

    return n, lines, centers

In [ ]:
def plot_lines_and_vp(im, lines, vp):
    """
    Plots user-input lines and the calculated vanishing point.
    Inputs:
        im: np.ndarray of shape (height, width, 3)
        lines: np.ndarray of shape (3, n)
            where each column denotes the parameters of the line equation
        vp: np.ndarray of shape (3, )
    """
    bx1 = min(1, vp[0] / vp[2]) - 10
    bx2 = max(im.shape[1], vp[0] / vp[2]) + 10
    by1 = min(1, vp[1] / vp[2]) - 10
    by2 = max(im.shape[0], vp[1] / vp[2]) + 10

    plt.figure()
    plt.imshow(im)
    for i in range(lines.shape[1]):
        if lines[0, i] < lines[1, i]:
            pt1 = np.cross(np.array([1, 0, -bx1]), lines[:, i])
            pt2 = np.cross(np.array([1, 0, -bx2]), lines[:, i])
        else:
            pt1 = np.cross(np.array([0, 1, -by1]), lines[:, i])
            pt2 = np.cross(np.array([0, 1, -by2]), lines[:, i])
        pt1 = pt1 / pt1[2]
        pt2 = pt2 / pt2[2]
        plt.plot([pt1[0], pt2[0]], [pt1[1], pt2[1]], 'g')

    plt.plot(vp[0] / vp[2], vp[1] / vp[2], 'ro')
    plt.show()

In [ ]:
def get_top_and_bottom_coordinates(im, obj):
    """
    For a specific object, prompts user to record the top coordinate and the bottom coordinate in the image.
    Inputs:
        im: np.ndarray of shape (height, width, 3)
        obj: string, object name
    Returns:
        coord: np.ndarray of shape (3, 2)
            where coord[:, 0] is the homogeneous coordinate of the top of the object and coord[:, 1] is the homogeneous
            coordinate of the bottom
    """
    plt.figure()
    plt.imshow(im)

    print('Click on the top coordinate of %s' % obj)
    clicked = plt.ginput(1, timeout=0, show_clicks=True)
    x1, y1 = clicked[0]
    # Uncomment this line to enable a vertical line to help align the two coordinates
    # plt.plot([x1, x1], [0, im.shape[0]], 'b')
    print('Click on the bottom coordinate of %s' % obj)
    clicked = plt.ginput(1, timeout=0, show_clicks=True)
    x2, y2 = clicked[0]

    plt.plot([x1, x2], [y1, y2], 'b')

    return np.array([[x1, x2], [y1, y2], [1, 1]])

# Your implementation

In [ ]:
def get_vanishing_point(lines):
    """
    Solves for the vanishing point using the user-input lines.
    
    Inputs:
        lines: np.ndarray of shape (3, n) where each column represents a line
               in homogeneous coordinates (a, b, c) where ax + by + c = 0
    
    Returns:
        vp: np.ndarray of shape (3,) representing the vanishing point in 
            homogeneous coordinates
    """
    # Method: Vanishing point is where all parallel lines meet
    # We solve this using least squares: Av = 0 where A is the matrix of lines
    # We want to find v that minimizes ||Av||^2
    
    # Use SVD to find the null space
    # lines.T is shape (n, 3), we want the right singular vector 
    # corresponding to smallest singular value
    
    _, _, Vt = np.linalg.svd(lines.T)
    vp = Vt[-1, :]  # Last row of Vt (last column of V)
    
    # Normalize so that the third coordinate is not too large
    # (helps with numerical stability)
    if np.abs(vp[2]) > 1e-8:
        vp = vp / vp[2]
    
    return vp

In [ ]:
def get_horizon_line(vp1, vp2):
    """
    Inputs:
        vp1, vp2 : 3-vectors (homogeneous image coords) of two *ground-plane* vanishing points
    Returns:
        l : 3-vector [a, b, c] for the line ax + by + c = 0, normalized so a^2 + b^2 = 1
    """
    import numpy as np
    v1 = np.asarray(vp1, dtype=float).ravel()
    v2 = np.asarray(vp2, dtype=float).ravel()

    # make sure both are homogeneous with w=1
    if abs(v1[2]) < 1e-12: v1[2] = 1.0
    if abs(v2[2]) < 1e-12: v2[2] = 1.0
    v1 = v1 / v1[2]
    v2 = v2 / v2[2]

    # horizon line = cross product of the two ground-plane vanishing points
    l = np.cross(v1, v2).astype(float)

    # normalize so a^2 + b^2 = 1
    n = (l[0]**2 + l[1]**2) ** 0.5
    if n > 0:
        l = l / n
    return l  # [a,b,c]

In [ ]:
def plot_horizon_line(im, horizon_line):
    """
    Plots the horizon line on the image.
    
    Inputs:
        im: np.ndarray of shape (height, width, 3)
        horizon_line: np.ndarray of shape (3,) - line parameters (a, b, c)
    """
    plt.figure()
    plt.imshow(im)
    
    # Line equation: a*x + b*y + c = 0
    # Solve for y: y = -(a*x + c) / b
    a, b, c = horizon_line
    
    if np.abs(b) > 1e-8:
        # Draw line across the image width
        x = np.array([0, im.shape[1]])
        y = -(a * x + c) / b
    else:
        # Vertical line case: x = -c/a
        x = np.array([-c/a, -c/a])
        y = np.array([0, im.shape[0]])
    
    plt.plot(x, y, 'r-', linewidth=2, label='Horizon Line')
    plt.legend()
    plt.title('Ground Horizon Line')
    plt.show()

In [ ]:
def get_camera_parameters(vp1, vp2, vp3, img_width, img_height):
    """
    Computes the camera parameters (f, u, v) from three orthogonal vanishing points.
    We assume:
        - zero skew
        - square pixels
        - principal point at the image center

    Inputs:
        vp1, vp2, vp3 : np.ndarray of shape (3,)  (homogeneous VPs)
        img_width, img_height : image dimensions

    Returns:
        f, u, v  (floats)
    """

    # Convert to inhomogeneous coordinates
    vpts = np.column_stack([vp1, vp2, vp3])   # shape (3,3)
    xs = vpts[0, :] / vpts[2, :]
    ys = vpts[1, :] / vpts[2, :]

    # Approximate principal point at image center
    u = img_width  / 2.0
    v = img_height / 2.0

    # Use the orthogonality constraint:
    # (x_i - u)(x_j - u) + (y_i - v)(y_j - v) + f^2 = 0
    pairs = [(0, 1), (0, 2), (1, 2)]
    f2_values = []

    for i, j in pairs:
        term = -((xs[i] - u) * (xs[j] - u) +
                 (ys[i] - v) * (ys[j] - v))
        f2_values.append(term)

    # Average the three estimates; clip to avoid negative due to noise
    f2 = max(np.mean(f2_values), 1e-6)
    f = math.sqrt(f2)

    print("Estimated intrinsics:")
    print(f"  f = {f:.2f} pixels")
    print(f"  u = {u:.2f} pixels")
    print(f"  v = {v:.2f} pixels")

    return f, u, v

In [ ]:
def get_rotation_matrix(vp1, vp2, vp3, f, u, v):
    """
    Computes the rotation matrix using the camera parameters.
    
    Inputs:
        vp1, vp2, vp3: vanishing points in homogeneous coordinates
        f, u, v: camera intrinsic parameters
    
    Returns:
        R: 3x3 rotation matrix
    """
    # <YOUR IMPLEMENTATION>
    
    # Construct camera intrinsic matrix K
    K = np.array([[f, 0, u],
                  [0, f, v],
                  [0, 0, 1]])
    
    K_inv = np.linalg.inv(K)
    
    # The columns of rotation matrix are the normalized directions
    # corresponding to the vanishing points: r_i = K^{-1} * v_i (normalized)
    
    r1 = K_inv @ vp1
    r1 = r1 / np.linalg.norm(r1)
    
    r2 = K_inv @ vp2
    r2 = r2 / np.linalg.norm(r2)
    
    r3 = K_inv @ vp3
    r3 = r3 / np.linalg.norm(r3)
    
    # Construct rotation matrix from column vectors
    R = np.column_stack([r1, r2, r3])
    
    # Ensure it's a proper rotation matrix using SVD
    # (projects to the nearest valid rotation matrix)
    U, _, Vt = np.linalg.svd(R)
    R = U @ Vt
    
    # Ensure determinant is +1 (not -1)
    if np.linalg.det(R) < 0:
        R = -R
    
    return R

In [ ]:
def estimate_height(ref_coords, obj_coords, ref_height, vp_vertical, horizon_line):
    """
    Estimates height for a specific object using the recorded coordinates. 
    You might need to plot additional images here for your report.
    
    Inputs:
        ref_coords: np.ndarray of shape (3, 2) - reference object [top, bottom] coords
        obj_coords: np.ndarray of shape (3, 2) - target object [top, bottom] coords
        ref_height: float - known height of reference object
        vp_vertical: np.ndarray of shape (3,) - vertical vanishing point
        horizon_line: np.ndarray of shape (3,) - horizon line parameters
    
    Returns:
        height: estimated height of the object
    """
    # <YOUR IMPLEMENTATION>
    
    # Extract coordinates
    ref_top = ref_coords[:, 0]
    ref_bottom = ref_coords[:, 1]
    obj_top = obj_coords[:, 0]
    obj_bottom = obj_coords[:, 1]
    
    # Method: Use cross-ratio to measure heights
    # The cross-ratio is preserved under projection
    
    def compute_cross_ratio(p1, p2, p3, p4):
        """
        Compute cross ratio of 4 collinear points.
        CR(A,B,C,D) = (AC/BC) / (AD/BD)
        """
        # Convert to inhomogeneous coordinates
        p1_2d = p1[:2] / p1[2]
        p2_2d = p2[:2] / p2[2]
        p3_2d = p3[:2] / p3[2]
        p4_2d = p4[:2] / p4[2]
        
        d13 = np.linalg.norm(p1_2d - p3_2d)
        d24 = np.linalg.norm(p2_2d - p4_2d)
        d14 = np.linalg.norm(p1_2d - p4_2d)
        d23 = np.linalg.norm(p2_2d - p3_2d)
        
        if d14 < 1e-8 or d23 < 1e-8:
            return 1.0
        
        return (d13 * d24) / (d14 * d23)
    
    # Find intersection of vertical line through reference with horizon
    ref_vertical_line = np.cross(ref_bottom, vp_vertical)
    ref_horizon_pt = np.cross(ref_vertical_line, horizon_line)
    if np.abs(ref_horizon_pt[2]) > 1e-8:
        ref_horizon_pt = ref_horizon_pt / ref_horizon_pt[2]
    
    # Find intersection of vertical line through object with horizon
    obj_vertical_line = np.cross(obj_bottom, vp_vertical)
    obj_horizon_pt = np.cross(obj_vertical_line, horizon_line)
    if np.abs(obj_horizon_pt[2]) > 1e-8:
        obj_horizon_pt = obj_horizon_pt / obj_horizon_pt[2]
    
    # Compute cross-ratios
    # CR(bottom, top, horizon_point, vanishing_point)
    cr_ref = compute_cross_ratio(ref_bottom, ref_top, ref_horizon_pt, vp_vertical)
    cr_obj = compute_cross_ratio(obj_bottom, obj_top, obj_horizon_pt, vp_vertical)
    
    # Height ratio from cross-ratios
    if abs(cr_ref) > 1e-8:
        height = ref_height * cr_obj / cr_ref
    else:
        # Fallback: simple pixel ratio
        height = ref_height * np.linalg.norm(obj_top[:2]/obj_top[2] - obj_bottom[:2]/obj_bottom[2]) / \
                            np.linalg.norm(ref_top[:2]/ref_top[2] - ref_bottom[:2]/ref_bottom[2])
    
    return height

# Main function

In [ ]:
im = np.asarray(Image.open('CSL.jpg'))

# Part 1
# Get vanishing points for each of the directions
num_vpts = 3
vpts = np.zeros((3, num_vpts))
for i in range(num_vpts):
    print('Getting vanishing point %d' % i)
    # Get at least three lines from user input
    n, lines, centers = get_input_lines(im)
    # <YOUR IMPLEMENTATION> Solve for vanishing point
    vpts[:, i] = get_vanishing_point(lines)
    print('Vanishing point %d: %s' % (i, vpts[:, i]))
    # Plot the lines and the vanishing point
    plot_lines_and_vp(im, lines, vpts[:, i])

# <YOUR IMPLEMENTATION> Get the ground horizon line
horizon_line = get_horizon_line(vpts[:, 0], vpts[:, 1])
print('\nHorizon line equation: %.6f*x + %.6f*y + %.6f = 0' % 
      (horizon_line[0], horizon_line[1], horizon_line[2]))
print('Verification: a^2 + b^2 = %.6f (should be 1.0)' % 
      (horizon_line[0]**2 + horizon_line[1]**2))

# <YOUR IMPLEMENTATION> Plot the ground horizon line
plot_horizon_line(im, horizon_line)

# Part 2
# <YOUR IMPLEMENTATION> Solve for the camera parameters (f, u, v)
f, u, v = get_camera_parameters(vpts[:, 0], vpts[:, 1], vpts[:, 2],
                                im.shape[1], im.shape[0])
print('\nCamera Parameters:')
print('Focal length f = %.2f pixels' % f)
print('Principal point: u = %.2f, v = %.2f' % (u, v))

# Part 3
# <YOUR IMPLEMENTATION> Solve for the rotation matrix
R = get_rotation_matrix(vpts[:, 0], vpts[:, 1], vpts[:, 2], f, u, v)
print('\nRotation Matrix:')
print(R)
print('det(R) = %.6f (should be 1.0)' % np.linalg.det(R))
print('Orthogonality check - R^T * R:')
print(R.T @ R)

# Part 4
# Record image coordinates for each object and store in map
objects = ('person', 'CSL building', 'the spike statue', 'the lamp posts')
coords = dict()
for obj in objects:
    coords[obj] = get_top_and_bottom_coordinates(im, obj)

# <YOUR IMPLEMENTATION> Estimate heights
# Assuming person is 5'6" = 66 inches
ref_height_inches = 66
vp_vertical = vpts[:, 2]  # Assuming third VP is vertical

print('\n=== Height Estimates (assuming person is 5ft 6in = 66 inches) ===')
for obj in objects[1:]:
    print('Estimating height of %s' % obj)
    height = estimate_height(coords['person'], coords[obj], 
                            ref_height_inches, vp_vertical, horizon_line)
    print('%s: %.2f inches = %.2f feet\n' % (obj, height, height/12))

# Re-estimate assuming person is 6ft = 72 inches
print('\n=== Height Estimates (assuming person is 6ft = 72 inches) ===')
ref_height_inches_6ft = 72
for obj in objects[1:]:
    height = estimate_height(coords['person'], coords[obj], 
                            ref_height_inches_6ft, vp_vertical, horizon_line)
    print('%s: %.2f inches = %.2f feet\n' % (obj, height, height/12))